## Task 3: Symmetric vs. Asymmetric INT8 Quantization

In [1]:
import numpy as np
import pandas as pd

In [2]:
weights = np.array([
    [-1.8, -0.9, 0.0, 0.7, 1.5],
    [-2.4, -0.3, 0.2, 1.1, 2.0]
], dtype=np.float32)

activations = np.array([
    [0.0, 0.3, 0.8, 1.4, 2.1],
    [0.1, 0.6, 1.0, 1.8, 3.2]
], dtype=np.float32)

outlier_tensor = np.array(
    [-0.5, -0.2, 0.0, 0.3, 0.7, 12.0],
    dtype=np.float32
)

### Part A

In [3]:
def symmetric_quantize(tensor):
    
    q_min = -127
    q_max = 127

    max_abs = np.max(np.abs(tensor))

    if max_abs == 0:
        scale = 1.0
    else:
        scale = max_abs / 127.0

    zero_point = 0

    q = np.round(tensor / scale)

    sat_min = np.sum(q < q_min)
    sat_max = np.sum(q > q_max)

    q = np.clip(q, q_min, q_max)

    return (
        q.astype(np.int8),
        scale,
        zero_point,
        sat_min,
        sat_max
    )

### Part B

In [4]:
def asymmetric_quantize(tensor):

    q_min = -128
    q_max = 127

    x_min = np.min(tensor)
    x_max = np.max(tensor)

    if np.isclose(x_max, x_min):
        scale = 1.0
        zero_point = 0
    else:
        scale = (x_max - x_min) / 255.0

        zero_point = round(
            -128 - (x_min / scale)
        )

        zero_point = int(
            np.clip(
                zero_point,
                q_min,
                q_max
            )
        )

    q = np.round(tensor / scale) + zero_point

    sat_min = np.sum(q < q_min)
    sat_max = np.sum(q > q_max)

    q = np.clip(
        q,
        q_min,
        q_max
    )

    return (
        q.astype(np.int8),
        scale,
        zero_point,
        sat_min,
        sat_max
    )

### Part C

In [5]:
def dequantize(
    quantized_tensor,
    scale,
    zero_point
):

    return (
        quantized_tensor.astype(np.float32)
        - zero_point
    ) * scale

In [6]:
def compute_metrics(
    original,
    reconstructed
):

    err = np.abs(
        original - reconstructed
    )

    mae = np.mean(err)

    mse = np.mean(
        (original - reconstructed) ** 2
    )

    max_err = np.max(err)

    return mae, mse, max_err, err

In [7]:
def evaluate_tensor(
    name,
    tensor
):

    results = []

    # Symmetric
    q_sym, s_sym, zp_sym, sat_min_sym, sat_max_sym = (
        symmetric_quantize(tensor)
    )

    dq_sym = dequantize(
        q_sym,
        s_sym,
        zp_sym
    )

    mae_sym, mse_sym, max_sym, _ = (
        compute_metrics(
            tensor,
            dq_sym
        )
    )

    results.append([
        name,
        "Symmetric",
        s_sym,
        zp_sym,
        mae_sym,
        mse_sym,
        max_sym,
        sat_min_sym,
        sat_max_sym,
        sat_min_sym + sat_max_sym
    ])

    # Asymmetric
    q_asym, s_asym, zp_asym, sat_min_asym, sat_max_asym = (
        asymmetric_quantize(tensor)
    )

    dq_asym = dequantize(
        q_asym,
        s_asym,
        zp_asym
    )

    mae_asym, mse_asym, max_asym, _ = (
        compute_metrics(
            tensor,
            dq_asym
        )
    )

    results.append([
        name,
        "Asymmetric",
        s_asym,
        zp_asym,
        mae_asym,
        mse_asym,
        max_asym,
        sat_min_asym,
        sat_max_asym,
        sat_min_asym + sat_max_asym
    ])

    return (
        results,
        q_sym,
        dq_sym,
        q_asym,
        dq_asym
    )

### Part D and E

In [8]:
all_results = []

for tensor_name, tensor in [
    ("Weights", weights),
    ("Activations", activations)
]:

    (
        results,
        q_sym,
        dq_sym,
        q_asym,
        dq_asym
    ) = evaluate_tensor(
        tensor_name,
        tensor
    )

    all_results.extend(results)

    print("\n" + "="*80)
    print(tensor_name)
    print("="*80)

    print("\nOriginal")
    print(tensor)

    print("\nSymmetric Quantized")
    print(q_sym)

    print("\nSymmetric Dequantized")
    print(dq_sym)

    print("\nAsymmetric Quantized")
    print(q_asym)

    print("\nAsymmetric Dequantized")
    print(dq_asym)


Weights

Original
[[-1.8 -0.9  0.   0.7  1.5]
 [-2.4 -0.3  0.2  1.1  2. ]]

Symmetric Quantized
[[ -95  -48    0   37   79]
 [-127  -16   11   58  106]]

Symmetric Dequantized
[[-1.7952756  -0.9070866   0.          0.6992126   1.4929134 ]
 [-2.4        -0.3023622   0.20787401  1.096063    2.0031495 ]]

Asymmetric Quantized
[[ -93  -41   11   52   98]
 [-128   -6   23   75  127]]

Asymmetric Dequantized
[[-1.7945098  -0.8972549   0.          0.707451    1.5011765 ]
 [-2.3984313  -0.29333332  0.20705882  1.1043137   2.0015686 ]]

Activations

Original
[[0.  0.3 0.8 1.4 2.1]
 [0.1 0.6 1.  1.8 3.2]]

Symmetric Quantized
[[  0  12  32  56  83]
 [  4  24  40  71 127]]

Symmetric Dequantized
[[0.        0.3023622 0.8062992 1.4110236 2.0913386]
 [0.1007874 0.6047244 1.007874  1.7889764 3.2      ]]

Asymmetric Quantized
[[-128 -104  -64  -16   39]
 [-120  -80  -48   15  127]]

Asymmetric Dequantized
[[0.         0.30117646 0.80313724 1.4054902  2.0956862 ]
 [0.10039216 0.6023529  1.0039215  1.

In [9]:
columns = [
    "Tensor",
    "Method",
    "Scale",
    "Zero Pt",
    "MAE",
    "MSE",
    "Max Err",
    "Sat(min)",
    "Sat(max)",
    "Sat(total)"
]

comparison_df = pd.DataFrame(
    all_results,
    columns=columns
)

comparison_df

,Tensor,Method,Scale,Zero Pt,MAE,MSE,Max Err,Sat(min),Sat(max),Sat(total)
0,Weights,Symmetric,0.018898,0,0.003701,0.000022,0.007874,0,0,0
1,Weights,Asymmetric,0.017255,11,0.003804,0.000021,0.007451,0,0,0
2,Activations,Symmetric,0.025197,0,0.005276,0.000045,0.011024,0,0,0
3,Activations,Asymmetric,0.012549,-128,0.002627,0.000011,0.005490,0,0,0


### Part F - Outlier Experiment

In [10]:
without_outlier = np.array(
    [-0.5, -0.2, 0.0, 0.3, 0.7],
    dtype=np.float32
)

outlier_results = []

for version, tensor in [
    ("With outlier", outlier_tensor),
    ("Without outlier", without_outlier)
]:

    # Symmetric
    q, s, zp, _, _ = (
        symmetric_quantize(tensor)
    )

    dq = dequantize(q, s, zp)

    mae, mse, max_err, err = (
        compute_metrics(
            tensor,
            dq
        )
    )

    print("\n" + "="*80)
    print(version, "- Symmetric")
    print("="*80)

    print("Original:")
    print(tensor)

    print("\nQuantized:")
    print(q)

    print("\nDequantized:")
    print(dq)

    print("\nPer-element Error:")
    print(err)

    outlier_results.append([
        version,
        "Symmetric",
        s,
        zp,
        mae,
        mse,
        max_err
    ])

    # Asymmetric
    q, s, zp, _, _ = (
        asymmetric_quantize(tensor)
    )

    dq = dequantize(q, s, zp)

    mae, mse, max_err, err = (
        compute_metrics(
            tensor,
            dq
        )
    )

    print("\n" + "="*80)
    print(version, "- Asymmetric")
    print("="*80)

    print("Original:")
    print(tensor)

    print("\nQuantized:")
    print(q)

    print("\nDequantized:")
    print(dq)

    print("\nPer-element Error:")
    print(err)

    outlier_results.append([
        version,
        "Asymmetric",
        s,
        zp,
        mae,
        mse,
        max_err
    ])


With outlier - Symmetric
Original:
[-0.5 -0.2  0.   0.3  0.7 12. ]

Quantized:
[ -5  -2   0   3   7 127]

Dequantized:
[-0.47244096 -0.18897638  0.          0.28346455  0.6614173  12.        ]

Per-element Error:
[0.02755904 0.01102363 0.         0.01653546 0.03858268 0.        ]

With outlier - Asymmetric
Original:
[-0.5 -0.2  0.   0.3  0.7 12. ]

Quantized:
[-128 -122 -118 -112 -104  127]

Dequantized:
[-0.49019608 -0.19607843  0.          0.29411766  0.6862745  12.009804  ]

Per-element Error:
[0.00980392 0.00392157 0.         0.00588235 0.01372546 0.00980377]

Without outlier - Symmetric
Original:
[-0.5 -0.2  0.   0.3  0.7]

Quantized:
[-91 -36   0  54 127]

Dequantized:
[-0.5015748 -0.1984252  0.         0.2976378  0.7      ]

Per-element Error:
[0.00157481 0.0015748  0.         0.00236222 0.        ]

Without outlier - Asymmetric
Original:
[-0.5 -0.2  0.   0.3  0.7]

Quantized:
[-128  -64  -22   42  127]

Dequantized:
[-0.49882355 -0.19764706  0.          0.3011765   0.7011765 ]

Outlier Comparison Table

In [11]:
outlier_df = pd.DataFrame(
    outlier_results,
    columns=[
        "Version",
        "Method",
        "Scale",
        "Zero Pt",
        "MAE",
        "MSE",
        "Max Error"
    ]
)

outlier_df

,Version,Method,Scale,Zero Pt,MAE,MSE,Max Error
0,With outlier,Symmetric,0.094488,0,0.015617,0.000441,0.038583
1,With outlier,Asymmetric,0.049020,-118,0.007190,0.000072,0.013725
2,Without outlier,Symmetric,0.005512,0,0.001102,0.000002,0.002362
3,Without outlier,Asymmetric,0.004706,-22,0.001176,0.000002,0.002353


OBSERVATIONS

1. Symmetric Quantization
   - Uses zero_point = 0.
   - Works best when data is centered around zero.
   - Commonly used for neural network weights.

2. Asymmetric Quantization
   - Uses both scale and zero_point.
   - Better utilizes the INT8 range when data is mostly positive.
   - Commonly used for activations.

3. Weights
   - Typically contain positive and negative values.
   - Symmetric quantization is usually sufficient.

4. Activations
   - Often non-negative.
   - Asymmetric quantization generally provides lower error.

5. Outlier Effect
   - A large outlier increases scale.
   - Larger scale reduces precision for normal values.
   - Removing the outlier usually decreases MAE and MSE significantly.

6. Practical Usage
   - Weights -> Symmetric INT8
   - Activations -> Asymmetric INT8